In [ ]:
# =============================================================================
# Created By:     Kai Metzger
# Created School: Franz-Oberthuer-Schule Wuerzburg
# Created Email:  metzgerkai@franz-oberthuer-schule.de
# Created Date:   Sat April 05 09:54 UTC 2025
# Changed Date:   Wed November 12 09:41 UTC 2026
# Version:        1.0.2
# =============================================================================
"""The Module has been build for training the symbols dataset with images + 
ground truth on a Raspberry Pi 5 with a standard USB camera. An image with a 
resolution of 640px x 480px can be recorded with the Python script 
create-symbols-dataset.py.
The files in the dataset <symbols> should be ordered in the following
manner:             > explanation
- symbols      
  - dataset1        > dataset version (dataset1 - 3 where used in the project)
    - data          > images (.png files)
    - gt            > ground truth (.txt files with class labels 1-4)
"""

# =============================================================================
# Import
# =============================================================================
import cv2
import os
import numpy as np
from keras import layers, models, utils, callbacks
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Enable Nvidia GPUs by un-commenting this line 
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
XLA_FLAGS="--xla_gpu_cuda_data_dir=/usr/"

# To get reproducable results with the same training setting random seed
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [ ]:
# =============================================================================
# Declare variables / Dataset
# =============================================================================
# Change the following paths to your dataset path
home_dir = os.path.expanduser("~")
dataset = 9                                                  # HYPERPARAMETER 1
dataset_for_training = "dataset" + str(dataset)            
img_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/data"
gt_dir = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/gt"
checkpoint_filepath = home_dir + "/ml_project/datasets/symbols/" + dataset_for_training + "/chpt/"

print(img_dir)

In [ ]:
# =============================================================================
# Declare variables / Dataset
# =============================================================================
img_size_x = 32                                            # HYPERPARAMETER 2.1
img_size_y = 32                                            # HYPERPARAMETER 2.2
channels = 3                                               # HYPERPARAMETER 2.3             
img_dim = img_size_x * img_size_y * channels

model = None
debug = True

In [ ]:
# =============================================================================
# Get number of data (image/label) 
# =============================================================================
anz_data = len(os.listdir(img_dir))
anz_data = int(anz_data)
dataset = np.zeros((anz_data, img_size_x, img_size_y, 3), dtype=np.float32)
ground_truth = np.zeros((anz_data), dtype=np.int32)
print("dataset size:", anz_data)

In [ ]:
# =============================================================================
# Define functions
# =============================================================================
# Change image size
def pic_prep(image, x, y):
    image = cv2.resize(image, (y, x))  # OpenCV expects (width, height)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image.astype(np.float32) / 255.0                     # NORMALIZATION

In [ ]:
# =============================================================================
# Callback for training
# =============================================================================
# Set verbose flag to 0 to omit loss/accuracy output per epoch
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath + "chpt.keras",
    save_weights_only=False,
    monitor='val_accuracy',
    mode='auto',
    save_best_only=True,
    verbose = 1)

In [ ]:
# =============================================================================
# Read dataset
# =============================================================================
for i in range(0, anz_data, 1):
    img_name = img_dir + '/' + str(i) + '.png'      # create file names
    image = cv2.imread(img_name)                    # read image
    image = pic_prep(image, img_size_x, img_size_y) # prepare image
    dataset[i] = image                          # 2d-image to 3d-array
    
    txt_name = gt_dir + '/' + str(i) + '.txt'       # create gt file names
    ground_truth[i] = np.genfromtxt(txt_name, delimiter=',') 

In [ ]:
# =============================================================================
# Prepare dataset: train set (80%), and val/test set (20%)
# =============================================================================
# dataset has shape (number_of_images, img_dim)
print("dataset shape:", dataset.shape)
labels = ground_truth.reshape(-1) # integer labels 0–3
print("ground_truth shape:", ground_truth.shape)

# 80% training, 20% temporary
train_data, temp_data, train_labels, temp_labels = train_test_split(
    dataset,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels,
)

# Split the temporary 20% equally: 10% validation, 10% test
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels,
)

# Convert integer labels to one-hot labels e.g. [0, 1, 0, 0]
# for categorical_crossentropy, on the contrary one could use
# sparse_categorical_crossentropy and keep the labels as integers.
train_gt = utils.to_categorical(train_labels, num_classes=4)
val_gt = utils.to_categorical(val_labels, num_classes=4)
test_gt = utils.to_categorical(test_labels, num_classes=4)

print("Train:", len(train_data), "Validation:", len(val_data), "Test:", len(test_data))

if False:
    print("Labels and counts:", np.unique(ground_truth, return_counts=True))
    print("Data:", train_data.shape, train_data.min(), train_data.max())
    print("One-hot labels:", train_gt.shape)

In [ ]:
# =============================================================================
# Apply horizontal flip augmentation to the training dataset
# =============================================================================
data_augmentation = keras.Sequential([                       # HYPERPARAMETER 8
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

In [ ]:
# =============================================================================
# Create neural network with x layers and (n_1, ..., n_i) neurons per layer.
# =============================================================================
# 1. Model
model = "small"
if model == "small":
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        layers.Lambda(tf.image.rgb_to_grayscale),
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),                               
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),        
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),                                                          
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.2),                                
        layers.Dense(4, activation="softmax"),
    ])
elif model == "medium":
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        layers.Lambda(tf.image.rgb_to_grayscale),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),                # HYPERPARAMETER 3.1
        layers.Dropout(0.3),                                # HYPERPARAMETER 3.2
        layers.Dense(512, activation="relu"),               # ...
        layers.Dropout(0.3),                               
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),                                # HYPERPARAMETER 3.x
        layers.Dense(4, activation="softmax"),
    ])
elif model == "cnn":
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        data_augmentation,
        #layers.Lambda(tf.image.rgb_to_grayscale),
        layers.Conv2D(16, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(4, activation="softmax"),
    ])
elif model == "cnn_custom":                     # ~ 70% accuracy
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        data_augmentation,
        #layers.Lambda(tf.image.rgb_to_grayscale),
        # Add Rescaling(1./255) here only if pixel values are in [0, 255].
        layers.Conv2D(16, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(4, activation="softmax"),
    ])
elif model ==  "lenet":
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        data_augmentation,

        # Add Rescaling(1./255) here if pixel values are in [0, 255].
        # For RGB input, you can optionally convert to grayscale:
        # layers.Lambda(tf.image.rgb_to_grayscale),

        layers.Conv2D(6, kernel_size=5, activation="tanh", padding="valid"),
        layers.AveragePooling2D(pool_size=2, strides=2),

        layers.Conv2D(16, kernel_size=5, activation="tanh", padding="valid"),
        layers.AveragePooling2D(pool_size=2, strides=2),

        layers.Flatten(),
        layers.Dense(120, activation="tanh"),
        layers.Dense(84, activation="tanh"),
        layers.Dense(4, activation="softmax"),
    ])
elif model == "vgg":
    model = models.Sequential([
        layers.Input(shape=(img_size_x, img_size_y, channels)),
        data_augmentation,

        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(4, activation="softmax"),
    ])

In [ ]:
# =============================================================================
# Take a look at our model
# =============================================================================
if debug:
    print(model.summary())

In [ ]:
# =============================================================================
# HYPERPARAMETERS
# =============================================================================
lr = "fixed"                                                 # HYPERPARAMETER 4
if lr == "fixed":
    learning_rate = 0.0001 
elif lr == "decay":
    learning_rate = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        decay_rate=0.9
    )                                      

optimizer = "adam"                                           # HYPERPARAMETER 5
if optimizer == "adam":
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
elif optimizer == "sgd":
    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
elif optimizer == "rmsprop":
    optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
elif optimizer == "adagrad":
    optimizer = keras.optimizers.Adagrad(learning_rate=learning_rate)

loss = "categorical_crossentropy"       
#loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)                    
#loss = "sparse_categorical_crossentropy"
#loss = "mean_squared_error"
#...

metric=["accuracy"]
#metric=["sparse_categorical_accuracy"]
#...

In [ ]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metric,
)

In [ ]:
# =============================================================================
# Train the neuronal network e.g. for 300 epochs.
# =============================================================================
history = model.fit(
    train_data,                                           
    train_gt,                                           
    batch_size=8,                                          # HYPERPARAMETER 6
    epochs=150,                                            # HYPERPARAMETER 7
    verbose=1,
    shuffle=True,
    validation_data=(val_data, val_gt),
    callbacks=[model_checkpoint_callback],
)

In [ ]:
# =============================================================================
# Plot train and val accuracy.
# =============================================================================

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# =============================================================================
# Plot train and val loss.
# =============================================================================

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='lower left')
plt.show()

In [ ]:
# =============================================================================
# Load pretrained dataset weights to e.g. test on new (unseen) data.
# =============================================================================
model.load_weights(checkpoint_filepath + "chpt.keras")

In [ ]:
# =============================================================================
# Test dataset on xxx.
# =============================================================================
score = model.evaluate(test_data, test_gt, verbose=1)
print('Test score:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# =============================================================================
# Testing on a single image.
# =============================================================================
img_path = (
    home_dir + "/ml_project/datasets/symbols/"
    + dataset_for_training + "/data/16.png"
)

img_pred = cv2.imread(img_path)
if img_pred is None:
    raise FileNotFoundError(f"Could not read image: {img_path}")

img_pred = pic_prep(img_pred, img_size_x, img_size_y)
data_pred = np.expand_dims(img_pred, axis=0)  # (1, 32, 32, 3)

result = model.predict(data_pred, verbose=0)

print("Probability for classes: (cross, circle, triangle, square) in percent", 
      result)

In [ ]:
# =============================================================================
# Output class: 
# translate class label (0,1,2,3) to class (cross, circle, triangle, square).
# =============================================================================

max_res = 0
res_index = 4
for i in range(0, 4, 1):
    if result[0,i] > max_res:
        max_res = result[0,i]
        res_index = i

if res_index == 0:
    print('Cross detected!')
elif res_index == 1:
    print('Circle detected!')
elif res_index == 2:
    print('Triangle detected!')
elif res_index == 3:
    print('Square detected!')
elif res_index == 4:
    print('Error!')